# Chapter 7: Multi-Head Attention

[Read this chapter online](https://jackluu.io/book/section-2-attention/ch07-multi-head-attention/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch07-multi-head-attention.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 7: Multi-Head Attention

![Where we are in the big picture](../assets/diagrams/ch07-where-we-are.png){ width="756" }
*Figure 7.1: We run multiple attention heads at the same time to gather richer context.*

In Chapter 6, we built one attention head. It acts as a single "reader" scanning the text. But one reader is rarely enough to catch everything. In this chapter you will:

- Run multiple attention heads in parallel.
- Understand how different heads learn to track different patterns.
- Concatenate their outputs into a single rich representation.

**Words to Know**
    - **Multi-Head Attention**: Running several attention operations simultaneously.
    - **Concatenation**: Joining multiple vectors end-to-end to form a longer vector.
    - **Projection**: A linear layer that blends the concatenated outputs.

## Theory

### One Reader Isn't Enough

![Parallel heads looking at different features](../assets/diagrams/ch07-parallel-heads.png){ width="458" }
*Figure 7.2: Every head sees the same input through its own filters, and their answers are joined rather than merged.*

When you read a sentence, you track multiple things at once:

- *Who* is the subject?
- *What action* is happening?
- *What is the tone*, serious or sarcastic?

A single attention head can only "look" for one kind of pattern at a time. **Multi-head attention** fixes this by running several heads in parallel, as shown in Figure 7.2. Think of it as a team of readers, each highlighting different details, who then combine all their notes.

### How It Works

Multi-head attention simply repeats the mechanism from Chapter 6:

1. Run 4 independent `SingleHeadAttention` modules on the exact same input.
2. Each head produces an output of 32 numbers (`head_size`).
3. Glue (concatenate) all 4 outputs together: `4 heads × 32 numbers = 128 numbers`.
4. Apply a final linear projection to blend them.

Because each head starts with different random weights (its own Q, K, and V filters), no two heads ever see the text the same way, and training pushes them further apart.

### Why Four Heads and Not One Big One

Four readers sound expensive. They are not, and this is the part that surprises people.

A head's filters are sized by its `head_size`, not by the full 128 channels. Split 128 channels across four heads and each head gets filters that are a quarter as wide. Multiply it out and the totals match exactly:

```python
head_size = C // n_heads

# Every head holds three filters (query, key, value), each C by head_size
four_heads = n_heads * 3 * C * head_size
one_head = 3 * C * C
```

```python
$ python src/examples/ch07_head_budget.py
4 heads of 32: 49,152 numbers
1 head of 128  : 49,152 numbers
Same budget: True
```

Four points of view cost the same as one. That is why every model in this family uses many heads: the split is free, so there is no reason to take one perspective when you can have four.

### Do the Heads Really Differ?

That is the claim. Here is the check, on the model you will train in Chapter 13.

The script below loads the trained weights, feeds in a real line of Shakespeare, and prints how much attention the **last** character pays to each earlier character, one row per head.

```python
for h, head in enumerate(model.blocks[0].attn.heads):
    with torch.no_grad():
        q, k = head.query(x), head.key(x)
        scores = q @ k.transpose(-2, -1) * head.head_size ** -0.5
        weights = F.softmax(scores, dim=-1)[0, -1]     # the last character's row
```

```python
$ python src/examples/ch07_heads_differ.py
Prompt: "JULIET: O Romeo"

Attention paid by the last character, one row per head:
              J    U    L    I    E    T    :    _    O    _    R    o    m    e    o
  head 0:  0.02 0.04 0.01 0.06 0.03 0.04 0.04 0.02 0.06 0.04 0.05 0.04 0.11 0.38 0.07
  head 1:  0.02 0.02 0.03 0.03 0.04 0.08 0.12 0.06 0.05 0.07 0.02 0.10 0.16 0.14 0.06
  head 2:  0.04 0.07 0.04 0.06 0.05 0.11 0.04 0.06 0.09 0.07 0.07 0.06 0.09 0.09 0.06
  head 3:  0.03 0.02 0.01 0.05 0.05 0.03 0.03 0.05 0.04 0.04 0.05 0.08 0.12 0.25 0.14

Sharpest focus per head: 0->'e' (0.38)  1->'m' (0.16)  2->'T' (0.11)  3->'e' (0.25)
```

Read the rows, not the labels. Head 0 commits: it puts 0.38 of its attention on the single character just before the end and largely ignores the rest. Head 3 does something similar but softer, splitting between `e` and the final `o`. Head 1 spreads itself over the colon, the `m` and the `e`, holding several places at once. Head 2 is nearly flat: its largest weight is 0.11 and its smallest is 0.04, which is close to paying equal attention to everything.

So the heads do differ, and not in the tidy way the textbook story suggests. One is sharp, one is soft, one is diffuse, and one is barely committing at all. That last one is worth sitting with: in a trained model, some heads do very little. Nobody assigned these roles, and nobody can promise that head 1 is the "grammar head". They are four different filters that started from four different random draws and were shaped by the same pressure to predict the next character.

**Watch Out**
    It is tempting to read a story into each head: this one tracks subjects, that one tracks punctuation. Sometimes a head really is that clean, and researchers have found interpretable ones in large models. Often it is not, as head 2 shows. Look at the numbers before you tell the story.

### The Output Projection

After concatenating the 4 heads, we pass the 128 numbers through one final linear layer (the `proj` layer). 

Why? The heads might have found redundant or conflicting information. The projection layer learns to blend the insights: "if head 1 and head 3 agree on this pattern, emphasize it; if head 2 is unsure, ignore it." It mixes the 4 separate perspectives into a single unified context vector of 128 numbers. This completes the "Attention" stage of our map (Figure 7.1).

There is a simpler thing we could have done here, and it is worth seeing why we did not. We could average the four heads instead of gluing them end to end. Averaging would give us 32 numbers rather than 128, which sounds tidy, but it throws away exactly what we paid for. Head 0's sharp focus on one character and head 2's flat spread would cancel each other into a lukewarm middle. Concatenation keeps every head's answer intact and lets the projection layer decide what each one is worth. Averaging decides in advance that they are all worth the same.

**In Business**
    ![Different departments form a single executive summary](../assets/diagrams/ch07-business-example.png){ width="298" }
    *Figure 7.3: Multi-head attention gathers different perspectives.*

    When building an assistant to draft emails in your company's house style, you don't just look for one pattern. You track tone, structure, and vocabulary (Figure 7.3). Multi-head attention works exactly the same way: it asks 4 different "departments" to evaluate the text, then compiles a final executive summary.

## Code

![How the code flows in multi-head attention](../assets/diagrams/ch07-code-flow.png){ width="458" }
*Figure 7.4: The heads run in parallel, get concatenated, and projected.*

We use `src/ch06_multihead_attention.py` to build the `MultiHeadAttention` class, as flow-charted in Figure 7.4.

```python
# Run each head in parallel
        head_outputs = [h(x) for h in self.heads]

        # Concatenate outputs along the last dimension
        out = torch.cat(head_outputs, dim=-1)

        # Apply projection and dropout
        out = self.dropout(self.proj(out))
```

Here is what happens when we run it:

```python
$ python src/ch06_multihead_attention.py
--- Comparing one head vs multi-head ---
Single head output  shape: torch.Size([2, 10, 32])  (head_size=32)
Multi-head output   shape: torch.Size([2, 10, 128])  (C=128)

Multi-head output has 4x more channels - it sees 4 perspectives at once.

Multi-head attention done! Ready for Chapter 8.
```

**What just happened:**

- Line 2 ran 4 independent attention heads at the same time.
- Line 2's result is that each head produced an output with 32 channels.
- Line 5 concatenated them, resulting in 128 channels (the original embedding dimension).
- Line 8 applied projection and dropout so the final output matches the exact shape of the input.

**Shape Check:** Table 7.1 outlines the shape transformations through the heads.

**Table 7.1:** Tensor shapes during the multi-head attention step.

| Variable | Shape | Meaning |
|---|---|---|
| `head_outputs` | 4 × `[B, T, 32]` | A list of outputs from the 4 heads. |
| `out` (after cat) | `[B, T, 128]` | The 4 outputs joined end-to-end. |
| `out` (final) | `[B, T, 128]` | The blended representation, ready for the next step. |

## Try It

**Try It**
    Change the number of heads in `src/utils/config.py`. Set `n_heads = 8` instead of 4. Run the script again. Notice how the `head_size` automatically drops to 16, so the final concatenated size is still 128 (8 × 16 = 128). The model can have more perspectives, but each one has less detail.

**Watch Out**
    For multi-head attention to work cleanly, your embedding dimension (`n_embd`) must be perfectly divisible by your number of heads (`n_heads`). If you try `n_embd = 128` and `n_heads = 5`, the program will crash because it cannot divide the channels equally.

## Key Takeaways

- Multi-head attention runs several single-head attention modules in parallel.
- Splitting the channels across heads costs nothing: four heads of 32 hold the same 49,152 numbers as one head of 128.
- The heads do end up different, but not in a tidy way. In our trained model one head is sharp, one is soft and one is close to flat.
- The outputs are concatenated, not averaged, so that no head's answer is diluted by the others before the projection can weigh it.
- A final linear projection blends the separate perspectives.
- The output shape `[B, T, C]` is exactly the same as the input shape, making it easy to stack layers.

## Check Your Understanding

1. Why is one attention head not enough to understand complex text?
2. If `n_embd = 256` and `n_heads = 8`, what is the `head_size`?
3. Four heads of 32 hold the same number of weights as one head of 128. Why does splitting cost nothing?
4. Why do we concatenate the heads rather than average them?
5. In the run above, head 2's attention was almost flat. What does that tell you about the claim that each head learns its own linguistic role?
6. What is the purpose of the final projection layer?


## Further Reading

**The architecture this book builds.** Reading a sequence one step at a time is slow, because step 500 cannot start until step 499 has finished, and distant words stay hard to connect. This paper removed the step-by-step reading entirely and kept only attention, plus a note of each token's position. Every token can then be processed at once, which is what made training on very large amounts of text practical. The model you build in Chapters 6 to 10 is this design, made small.

<div class="refs" markdown>

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). *Attention is all you need* (arXiv:1706.03762). arXiv. https://doi.org/10.48550/arXiv.1706.03762

</div>

---

### `src/ch06_multihead_attention.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch06_multihead_attention.py"   # a cell has none, and the file uses it to find the text

"""
Implement multi-head self-attention.
This file belongs to Chapter 7.
Run: python src/ch06_multihead_attention.py
"""
import torch
import torch.nn as nn

import os
import sys

from src.utils.config import GPTConfig
from src.ch05_self_attention import SingleHeadAttention

# Settings
config = GPTConfig()

# --- The Idea ---
class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = config.n_embd // config.n_heads

        # Create multiple independent attention heads
        self.heads = nn.ModuleList([
            SingleHeadAttention(head_size) for _ in range(config.n_heads)
        ])

        # Linear layer to project the concatenated head outputs back
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # Run each head in parallel
        head_outputs = [h(x) for h in self.heads]

        # Concatenate outputs along the last dimension
        out = torch.cat(head_outputs, dim=-1)

        # Apply projection and dropout
        out = self.dropout(self.proj(out))
        return out

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 7: Multi-Head Attention\n")

    mha = MultiHeadAttention()

    total_params = sum(p.numel() for p in mha.parameters())
    print(f"Config: n_heads={config.n_heads}, "
          f"head_size={config.n_embd // config.n_heads}")
    print(f"MultiHeadAttention total parameters: {total_params:,}")

    head_params = sum(p.numel() for h in mha.heads for p in h.parameters())
    proj_params = sum(p.numel() for p in mha.proj.parameters())
    print(f"  From {config.n_heads} heads: {head_params:,}")
    print(f"  From output proj : {proj_params:,}")

    B, T = 2, 10
    x = torch.randn(B, T, config.n_embd)
    out = mha(x)

    print(f"\nInput  shape: {x.shape}")
    print(f"Output shape: {out.shape}   (same shape as input!)")

    print("\n--- Comparing one head vs multi-head ---")
    single_head = SingleHeadAttention(config.n_embd // config.n_heads)

    # Use no_grad to skip tracking operations since we don't need backprop
    with torch.no_grad():
        single_out = single_head(x)
        multi_out  = mha(x)

    print(f"Single head output  shape: {single_out.shape}  (head_size=32)")
    print(f"Multi-head output   shape: {multi_out.shape}  (C=128)")
    ratio = multi_out.shape[-1] // single_out.shape[-1]
    print(f"\nMulti-head output has {ratio}x "
          f"more channels - it sees {config.n_heads} perspectives at once.")

    print("\nMulti-head attention done! Ready for Chapter 8.")

---

### `src/examples/ch07_head_budget.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch07_head_budget.py"   # a cell has none, and the file uses it to find the text

"""
Count the numbers inside the heads: four small heads against one big one.
This file belongs to Chapter 7.
Run: python src/examples/ch07_head_budget.py
"""
C, n_heads = 128, 4                      # channels, and how many heads we split them into
head_size = C // n_heads

# Every head holds three filters (query, key, value), each C by head_size
four_heads = n_heads * 3 * C * head_size
one_head = 3 * C * C

print(f"{n_heads} heads of {head_size}: {four_heads:,} numbers")
print(f"1 head of {C}  : {one_head:,} numbers")
print(f"Same budget: {four_heads == one_head}")

---

### `src/examples/ch07_heads_differ.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch07_heads_differ.py"   # a cell has none, and the file uses it to find the text

"""
Show that the trained model's attention heads look at different characters.
This file belongs to Chapter 7, and needs the model trained in Chapter 13.
Run: python src/examples/ch07_heads_differ.py
"""
import os
import sys

import torch
import torch.nn.functional as F

from src.ch03_tokenizer import build_vocab, encode
from src.ch09_gpt_model import GPT
from src.utils.config import GPTConfig

PROMPT = "JULIET: O Romeo"
CHECKPOINT = "checkpoints/model.pt"

if not os.path.exists(CHECKPOINT):
    sys.exit("No trained model yet. Train one first: python src/ch12_train.py")

text = open("src/data/shakespeare.txt", encoding="utf-8").read()
_, char_to_id, _ = build_vocab(text)

model = GPT(GPTConfig())
# weights_only=False because the checkpoint also stores the config object
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()

# What the first block sees: token vectors plus position vectors, normalized
ids = torch.tensor([encode(PROMPT, char_to_id)])
x = model.blocks[0].ln1(model.token_emb(ids) + model.pos_emb(torch.arange(ids.shape[1])))

print(f'Prompt: "{PROMPT}"\n')
print("Attention paid by the last character, one row per head:")
print("          " + "".join(f"{c if c != ' ' else '_':>5}" for c in PROMPT))

sharpest = []
for h, head in enumerate(model.blocks[0].attn.heads):
    with torch.no_grad():
        scores = head.query(x) @ head.key(x).transpose(-2, -1) * head.head_size ** -0.5
        weights = F.softmax(scores, dim=-1)[0, -1]     # the last character's row
    print(f"  head {h}: " + "".join(f"{w:5.2f}" for w in weights.tolist()))
    sharpest.append(f"{h}->'{PROMPT[weights.argmax()]}' ({weights.max():.2f})")

print("\nSharpest focus per head: " + "  ".join(sharpest))